# Ders 11 - Model Bağlam Protokolü (MCP)

**Model Bağlam Protokolü (MCP)**, ajanların çalışma zamanında araçları, kaynakları ve veri kaynaklarını dinamik olarak keşfetmesini ve kullanmasını sağlayan açık bir standarttır. Araçları bir ajana sabit kodlamak yerine, MCP ajanların yetenekleri talep üzerine sunan harici sunuculara bağlanmasına izin verir.

Bu derste, şunları öğreneceksiniz:
- MCP'nin ne olduğu ve ajan sistemleri için neden önemli olduğu
- MCP istemci-sunucu mimarisinin nasıl çalıştığı
- MCP tarzı araç keşfi kullanan ajanların nasıl inşa edileceği


## Kurulum

**Önkoşullar:**
- Dağıtılmış bir modele sahip Microsoft Foundry projesi
- Kimlik doğrulama için `az login` komutunu çalıştırın


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Model Context Protocol (MCP) Nedir?

MCP, yapay zeka ajanlarının dış araçları ve veri kaynaklarını keşfetmesi ve etkileşime geçmesi için standart bir yol tanımlar:

- **MCP Sunucusu**: Araçları, kaynakları ve istemleri standart bir protokol üzerinden sunar
- **MCP İstemcisi**: Sunuculara bağlanan ve mevcut yetenekleri keşfeden ajan çalışma zamanı
- **Dinamik Keşif**: Ajanların önceden tanımlanmış araçlara ihtiyacı yoktur — çalışma zamanında nelerin mevcut olduğunu keşfederler

Bu, ajan kodunu değiştirmeden yeni yetenekler eklenebilen genişletilebilir ajan sistemleri oluşturmak için güçlüdür.


## MCP Nasıl Çalışır

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ajan (MCP istemcisi) bir MCP sunucusuna bağlanır
2. Sunucu, kullanılabilir araçlar ve şemaları listesini yanıtlar
3. Ajan daha sonra mantık yürütmesi sırasında keşfedilen herhangi bir aracı çağırabilir
4. Sonuçlar aynı protokol üzerinden geri akar


## MCP Aracı Keşfini Simüle Etme

Gerçek bir MCP sunucusu çalışan bir sunucu süreci gerektirdiği için, bir MCP bağlantılı konaklama hizmetinin sağlayacaklarını taklit eden `@tool` işlevlerini kullanarak deseni göstereceğiz.

Üretimde, bu araçlar yerel olarak tanımlanmak yerine dinamik olarak bir MCP sunucusundan keşfedilir.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP Tarzı Araçlarla Bir Ajan İnşa Etmek


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## Üretimde MCP

Bir üretim ortamında, MCP güçlü desenleri mümkün kılar:

- **Dinamik araç keşfi**: Ajanlar MCP sunucularına bağlanır ve çalışma zamanında araçları keşfeder
- **Bağımsız mimari**: Araç sağlayıcıları, ajandan bağımsız olarak güncellenebilir
- **Kuruluşlar arası paylaşım**: Takımlar, herhangi bir ajanın kullanabileceği yetenekleri MCP sunucuları aracılığıyla açabilir
- **Microsoft Agent Framework desteği**: MAF, `mcp` entegrasyonu ile yerleşik MCP istemci desteğine sahiptir

Gerçek bir MCP sunucusunu MAF ile kullanmak için `hosted_mcp_tool()` veya MCP istemci entegrasyonu üzerinden bağlanırsınız.

**Daha fazla bilgi:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Özet

Bu derste şunları öğrendiniz:
- **MCP**, ajanlar ile araç sağlayıcıları arasında dinamik araç keşfi için açık bir standarttır
- **İstemci-sunucu mimarisi**, ajanların çalışma zamanında yetenekleri keşfetmesini sağlar
- MCP, araçların kod değişikliği olmadan eklenebildiği **genişletilebilir, ayrı bağlı ajan sistemleri** sağlar
- Microsoft Agent Framework, üretim kullanımı için **yerleşik MCP desteği** sunar


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
